In [1]:
# Requires vllm>=0.8.5
from vllm import LLM
import json

with open('./data/corpus.json', 'r', encoding='utf-8') as f:
    CSAID_corpus_list = json.load(f)
with open("./data/eval.json", "r") as f:
    CSAID_eval_data = json.load(f)

Model_Path = "/data/shared/users/liyunhan/Model/Qwen/Qwen3-Embedding-4B"
model = LLM(model=Model_Path)

INFO 03-05 21:36:42 [utils.py:223] non-default args: {'disable_log_stats': True, 'model': '/data/shared/users/liyunhan/Model/Qwen/Qwen3-Embedding-4B'}
INFO 03-05 21:36:42 [config.py:764] Found sentence-transformers modules configuration.
INFO 03-05 21:36:42 [config.py:791] Found pooling configuration.
INFO 03-05 21:36:42 [model.py:801] Resolved `--runner auto` to `--runner pooling`. Pass the value explicitly to silence this message.
INFO 03-05 21:36:42 [model.py:853] Resolved `--convert auto` to `--convert embed`. Pass the value explicitly to silence this message.
INFO 03-05 21:36:42 [model.py:529] Resolved architecture: Qwen3ForCausalLM
INFO 03-05 21:36:42 [model.py:1549] Using max model len 40960
INFO 03-05 21:36:43 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-05 21:36:43 [vllm.py:689] Asynchronous scheduling is enabled.
WARNING 03-05 21:36:43 [vllm.py:824] Pooling models do not support full cudagraphs. Overriding cudagraph_mode to PIECEWISE

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore_DP0 pid=3328955) INFO 03-05 21:36:47 [adapters.py:204] Mapping weights to Qwen3Model as they are relative to this model instead of Qwen3ForEmbedding.
(EngineCore_DP0 pid=3328955) INFO 03-05 21:36:50 [default_loader.py:293] Loading weights took 2.60 seconds
(EngineCore_DP0 pid=3328955) INFO 03-05 21:36:50 [gpu_model_runner.py:4221] Model loading took 7.55 GiB memory and 3.245961 seconds
(EngineCore_DP0 pid=3328955) INFO 03-05 21:36:59 [backends.py:916] Using cache directory: /home/liyunhan/.cache/vllm/torch_compile_cache/2dbf1aa20c/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3328955) INFO 03-05 21:36:59 [backends.py:976] Dynamo bytecode transform time: 8.60 s
(EngineCore_DP0 pid=3328955) INFO 03-05 21:37:05 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.987 s
(EngineCore_DP0 pid=3328955) INFO 03-05 21:37:05 [monitor.py:34] torch.compile takes 10.58 s in total
(EngineCore_DP0 pid=3328955) INFO 03-05 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 16.33it/s]


(EngineCore_DP0 pid=3328955) INFO 03-05 21:37:10 [gpu_model_runner.py:5246] Graph capturing finished in 4 secs, took 0.38 GiB
(EngineCore_DP0 pid=3328955) INFO 03-05 21:37:10 [core.py:278] init engine (profile, create kv cache, warmup model) took 19.46 seconds
INFO 03-05 21:37:11 [llm.py:355] Supported tasks: ['token_embed', 'embed']


In [2]:
import json
import re

max_length = 40000
pattern = re.compile(r'>,\s*(第)?[^:\n\r]*(章|节|章节)[:：]')

new_CSAID_corpus_list = []

for text in CSAID_corpus_list:
    # 命中章节标题（章 / 节 / 章节，但不是条）→ 直接丢弃
    if pattern.search(text):
        continue

    # 长度截断
    if len(text) > max_length:
        text = text[:max_length]

    new_CSAID_corpus_list.append(text)

CSAID_corpus_list = new_CSAID_corpus_list

print("Load corpus, length:",len(CSAID_corpus_list))


Load corpus, length: 69874


In [3]:
import os
import numpy as np
import torch

cache_path = "./embeddings.npy"

# ===================== 读取缓存 =====================
if os.path.exists(cache_path):
    print("[Load] exist embedding")
    embeddings = torch.from_numpy(np.load(cache_path))

# ===================== 重新生成 =====================
else:
    print("[Build] generating embeddings")

    outputs = model.embed(CSAID_corpus_list, use_tqdm=True)

    embs = np.array(
        [o.outputs.embedding for o in outputs],
        dtype=np.float32
    )

    np.save(cache_path, embs)
    embeddings = torch.from_numpy(embs)
emb_np = embeddings.cpu().numpy()
print("Embedding shape:", embeddings.shape)
print("✅ Finished")

[Load] exist embedding
Embedding shape: torch.Size([69874, 2560])
✅ Finished


In [4]:
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

def embed_query(query: str, Instruct: str = "Given a web query, retrieve relevant passages that answer the query."):
    instruct = f"Instruct: {Instruct}\nQuery: {query}"
    outputs = model.embed([instruct], use_tqdm=False)
    return torch.tensor([o.outputs.embedding for o in outputs])

def search_scores(query: str, top_k_embedding=60, instruct = "Given a web query, retrieve relevant passages that answer the query."):

    q_emb = embed_query(query, instruct)
    q_emb_np = q_emb.cpu().numpy()

    sims = np.dot(emb_np, q_emb_np.T).flatten()
    top_indices = np.argsort(sims)[-top_k_embedding:][::-1]
    
    docs = [CSAID_corpus_list[i] for i in top_indices]
    scores = [float(sims[i]) for i in top_indices]
    return docs, scores

query = "杀人犯法嘛?"
docs, scores = search_scores(query, top_k_embedding=60)
print(docs)

['<中华人民共和国刑法(2023修正)>, 第二百三十二条:\n【故意杀人罪】故意杀人的，处死刑、无期徒刑或者十年以上有期徒刑；情节较轻的，处三年以上十年以下有期徒刑。', '<中华人民共和国刑法(2023修正)>, 第四十九条:\n【死刑适用对象的限制】犯罪的时候不满十八周岁的人和审判的时候怀孕的妇女，不适用死刑。审判的时候已满七十五周岁的人，不适用死刑，但以特别残忍手段致人死亡的除外。', '<中华人民共和国民法典>, 第一千零二条:\n自然人享有生命权。自然人的生命安全和生命尊严受法律保护。任何组织或者个人不得侵害他人的生命权。', '<中华人民共和国刑法(2023修正)>, 第四十八条:\n【死刑、死缓的适用对象及核准程序】死刑只适用于罪行极其严重的犯罪分子。对于应当判处死刑的犯罪分子，如果不是必须立即执行的，可以判处死刑同时宣告缓期二年执行。死刑除依法由最高人民法院判决的以外，都应当报请最高人民法院核准。死刑缓期执行的，可以由高级人民法院判决或者核准。', '<最高人民法院关于适用《中华人民共和国民法典》继承编的解释(一)>, 第七条:\n继承人故意杀害被继承人的，不论是既遂还是未遂，均应当确认其丧失继承权。', '<中华人民共和国刑法(2023修正)>, 第十四条:\n【故意犯罪】明知自己的行为会发生危害社会的结果，并且希望或者放任这种结果发生，因而构成犯罪的，是故意犯罪。故意犯罪，应当负刑事责任。', '<关于办理死刑案件审查判断证据若干问题的规定>, 第十七条:\n对被害人陈述的审查与认定适用前述关于证人证言的有关规定。4、被告人供述和辩解', '<中华人民共和国刑事诉讼法(2018修正)>, 第二百四十六条:\n死刑由最高人民法院核准。', '<关于办理死刑案件审查判断证据若干问题的规定>, 第五条:\n办理死刑案件，对被告人犯罪事实的认定，必须达到证据确实、充分。证据确实、充分是指：（一）定罪量刑的事实都有证据证明；（二）每一个定案的证据均已经法定程序查证属实；（三）证据与证据之间、证据与案件事实之间不存在矛盾或者矛盾得以合理排除；（四）共同犯罪案件中，被告人的地位、作用均已查清；（五）根据证据认定案件事实的过程符合逻辑和经验规则，由证据得出的结论为唯一结论。办理死刑案件，对于以下事实的证明必须达到证据确实、充分：（一）被指控的犯罪事实的发生；

In [5]:
from tqdm import tqdm
import math

def calculate_metrics(ranked_contents, label_list, k=10):
    """
    计算RAG检索评估指标，包括 NDCG@K
    
    Args:
        ranked_contents: 检索结果列表（按排序顺序）
        label_list: 相关文档ID列表（标准答案）
        k: top-k值
    
    Returns:
        dict: 包含各项指标的字典
    """
    if not ranked_contents:
        return {
            'MRR': 0.0,
            'Recall': 0.0,
            'Precision': 0.0,
            'F1': 0.0,
            'HitRate': 0,
            'NDCG': 0.0,
            'relevant_count': 0,
            'total_relevant': len(set(label_list)),
            'k': k
        }

    # 确保k不超过检索结果数量
    k = min(k, len(ranked_contents))
    
    # 转换为集合便于快速查找
    relevant_set = set(label_list)
    top_k_docs = ranked_contents[:k]
    
    # 计算相关文档数量
    relevant_count = sum(1 for doc in top_k_docs if doc in relevant_set)
    total_relevant = len(relevant_set)
    
    # 1. MRR (Mean Reciprocal Rank)
    mrr = 0.0
    for i, doc in enumerate(ranked_contents, 1):
        if doc in relevant_set:
            mrr = 1.0 / i
            break
    
    # 2. Recall@K
    recall = relevant_count / total_relevant if total_relevant > 0 else 0
    
    # 3. Precision@K
    precision = relevant_count / k if k > 0 else 0
    
    # 4. F1@K
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # 5. Hit率@K
    hit_rate = 1 if relevant_count > 0 else 0

    # 6. NDCG@K (binary relevance: rel=1 if in label_list, else 0)
    def dcg_at_k(ranked_docs, relevant_set, k):
        dcg = 0.0
        for i in range(k):
            doc = ranked_docs[i]
            rel = 1 if doc in relevant_set else 0
            dcg += rel / math.log2(i + 2)  # log2(rank + 1), rank starts at 1
        return dcg

    # Ideal DCG: 将所有相关文档排在最前面
    ideal_ranking = list(relevant_set)[:k] + ['dummy'] * (k - min(len(relevant_set), k))
    idcg = dcg_at_k(ideal_ranking, relevant_set, k)

    if idcg == 0:
        ndcg = 0.0
    else:
        actual_dcg = dcg_at_k(ranked_contents, relevant_set, k)
        ndcg = actual_dcg / idcg

    return {
        'MRR': mrr,
        'Recall': recall,
        'Precision': precision,
        'F1': f1,
        'HitRate': hit_rate,
        'NDCG': ndcg,
        'relevant_count': relevant_count,
        'total_relevant': total_relevant,
        'k': k
    }

def Evaluation_CSAID(CSAID_eval_data, top_k_embedding = 60,  max_num_of_test=999):
    # 初始化总和变量
    total_mrr = 0
    total_recall = 0
    total_precision = 0
    total_f1 = 0
    total_hit_rate = 0
    total_ndcg = 0
    query_list = list(CSAID_eval_data.keys())

    total_queries = min(len(query_list),max_num_of_test)


    # 遍历所有查询
    for query in tqdm(query_list):
        docs, scores = search_scores(query, top_k_embedding = top_k_embedding)
        label_list = CSAID_eval_data[query]['Final_label']
        
        metrics = calculate_metrics(docs, label_list, k=len(docs))
        
        total_mrr += metrics['MRR']
        total_recall += metrics['Recall']
        total_precision += metrics['Precision']
        total_f1 += metrics['F1']
        total_hit_rate += metrics['HitRate']
        total_ndcg += metrics['NDCG']


    # 计算平均值
    avg_mrr = total_mrr / total_queries
    avg_recall = total_recall / total_queries
    avg_ndcg = total_ndcg / total_queries
    avg_precision = total_precision / total_queries
    avg_f1 = total_f1 / total_queries
    avg_hit_rate = total_hit_rate / total_queries

    metrics_dict = {
        "MRR" : avg_mrr,
        "Recall" : avg_recall,
        "nDCG" : avg_ndcg,
        "Precision" : avg_precision,
        "F1" : avg_f1,
        "HitRate" : avg_hit_rate
    }

    return metrics_dict

In [6]:
Deli_metrics_dict = Evaluation_CSAID(CSAID_eval_data)
print(Deli_metrics_dict)


100%|██████████| 118/118 [00:04<00:00, 24.71it/s]

{'MRR': 0.7274646257697105, 'Recall': 0.7601077636222202, 'nDCG': 0.5823592565155992, 'Precision': 0.0861581920903954, 'F1': 0.15162879274221497, 'HitRate': 0.9830508474576272}
